# RAG Agent Evaluation

In this notebook, we'll evaluate the IT Help Desk RAG agent from Module 2 using comprehensive evaluation metrics. You'll learn how to measure agent quality using RAGAS metrics and LLM-as-a-judge techniques with NVIDIA models.

## Setup
The container has the required packages. Load the project configuration and open Workshop Health if a service or import is unavailable.

In [ ]:
# The Workbench image includes the workshop dependencies. Check Workshop Health if imports fail.
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".project/spec.yaml").exists())
MODULE_DIR = PROJECT_ROOT / "code/3-agent-evaluation"
sys.path.insert(0, str(PROJECT_ROOT / "code"))
sys.path.insert(0, str(MODULE_DIR))
from workshop_support import get_model, load_secrets
load_secrets(PROJECT_ROOT)

## Import Libraries

Import all the necessary libraries for evaluation:

In [ ]:
import json
import importlib
# Pick up rubric/model edits when rerunning in the same kernel.
if "evaluation_framework" in sys.modules:
    importlib.reload(sys.modules["evaluation_framework"])
import pandas as pd
from evaluation_framework import create_judge_llm, create_embeddings, evaluate_rag_response, parse_evaluation, EvaluationResult, JUDGE_MODEL, JUDGE_MAX_TOKENS, JUDGE_MODEL_KWARGS, JUDGE_REQUESTS_PER_SECOND, JUDGE_TIMEOUT, JUDGE_RETRY_ATTEMPTS, EMBEDDING_MODEL
from evaluation_support import (final_answer, retrieval_trace, trace_contexts, citation_check, error_summary,
    invoke_with_retry, store_scores, skip_scores, score_text, quality_summary, fingerprint, start_run, checkpoint, select_dataset, compare_runs, record_config)
METRICS = ["faithfulness", "relevancy", "helpfulness"]

## Load a fixed test dataset
Each case contains a question, reference answer, retrieval keywords, and category. References are evidence to review, not automatic truth. Keep the same saved dataset for a before/after comparison.

### Exercise
Load the JSON array into `test_dataset`.

<details>
<summary>💡 NEED SOME HELP?</summary>

```python
test_dataset = json.load(f)
```
</details>

In [ ]:
data_dir = PROJECT_ROOT / "data/evaluation"
test_dataset_path = select_dataset(data_dir, "rag_agent")
with test_dataset_path.open() as f:
    # TODO: Load the JSON array.
    test_dataset = ...
assert test_dataset, "The dataset is empty."
print(f"Loaded {len(test_dataset)} cases from {test_dataset_path.name}")
print("Review generated references against their source articles before drawing quality conclusions.")

## Initialize the RAG agent
This evaluation uses your Module 2 model, retriever tool, and `SYSTEM_PROMPT`, with only the KB tool enabled. Its results describe that KB-only configuration. Editing Module 2’s prompt now changes the evaluated prompt.

Complete Module 2’s splitter, embedding, reranker, and model exercises first; MCP and Skills are optional here. Restart the kernel after editing imported Python files.

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / "code/2-agentic-rag"))
if "rag_agent" in sys.modules:
    importlib.reload(sys.modules["rag_agent"])
import rag_agent
from langgraph.prebuilt import create_react_agent

assert rag_agent.llm is not Ellipsis, "Finish Module 2's llm exercise."
AGENT = create_react_agent(model=rag_agent.llm, tools=[rag_agent.RETRIEVER_TOOL], prompt=rag_agent.SYSTEM_PROMPT)
config = {"agent_model": rag_agent.llm.model, "judge_model": JUDGE_MODEL,
          "embedding_model": rag_agent.embeddings.model, "reranking_model": rag_agent.reranker.model,
          "system_prompt": rag_agent.SYSTEM_PROMPT, "tools": [rag_agent.RETRIEVER_TOOL.name],
          "chunk_size": rag_agent.splitter._chunk_size, "chunk_overlap": rag_agent.splitter._chunk_overlap,
          "retrieval_settings": rag_agent.kb_retriever.search_kwargs, "rerank_top_n": rag_agent.reranker.top_n,
          "agent_temperature": rag_agent.llm.temperature, "judge_temperature": 0, "judge_max_tokens": JUDGE_MAX_TOKENS, "judge_model_kwargs": JUDGE_MODEL_KWARGS, "judge_requests_per_second": JUDGE_REQUESTS_PER_SECOND, "judge_timeout": JUDGE_TIMEOUT, "judge_retry_attempts": JUDGE_RETRY_ATTEMPTS,
          "rubric_sha256": fingerprint((MODULE_DIR / "evaluation_framework.py").read_text()),
          "knowledge_base_sha256": fingerprint({p.name: p.read_text() for p in sorted(rag_agent.DATA_DIR.glob("*.md"))})}
run_dir = start_run("rag", test_dataset_path, config, MODULE_DIR)
print("Run:", run_dir.name)

If importing Module 2 fails, distinguish an unfinished `...` exercise from an API or dependency error. Workshop Health checks those services separately. Saved run manifests record the dataset hash, prompt, model IDs, retrieval settings, and package versions.

## Run the agent
### Exercise
Invoke the agent on each question and copy the case’s question, reference, and category into the result row. Provided helpers extract the final answer and **actual tool artifacts**. They retain query rewrites and repeated retrieval calls; there is no second retrieval for evaluation. Errors and empty answers are recorded separately.

<details>
<summary>💡 NEED SOME HELP?</summary>

```python
response = invoke_with_retry(lambda: AGENT.invoke({
            "messages": [{"role": "user", "content": test_case["question"]}]
        }))
```
</details>

<details>
<summary>💡 NEED SOME HELP?</summary>

```python
"question": test_case["question"], "ground_truth": test_case["ground_truth"],
           "category": test_case["category"]
```
</details>

In [ ]:
results = []
for i, test_case in enumerate(test_dataset):
    row = {"case_id": f"{i:03d}-{fingerprint(test_case)[:12]}",
           # TODO: Copy these three fields from the test case.
           "question": ..., "ground_truth": ...,
           "category": ...,
           "agent_response": "", "retrieved_contexts": [], "retrieval_trace": []}
    if any(row[k] is Ellipsis for k in ("question", "ground_truth", "category")):
        raise NotImplementedError("Complete the result-row fields before running the agent.")
    try:
        # TODO: Invoke the agent with the question as a user message.
        response = ...
        if response is Ellipsis:
            raise NotImplementedError("Complete the agent invocation exercise.")
        row["agent_response"] = final_answer(response)
        row["retrieval_trace"] = retrieval_trace(response)
        row["retrieved_contexts"] = trace_contexts(row["retrieval_trace"])
        row["agent_status"] = "ok" if row["agent_response"] else "empty_response"
    except NotImplementedError:
        raise
    except Exception as exc:
        row.update(agent_status="error", agent_error=error_summary(exc))
    results.append(row)
    checkpoint(run_dir, results)
    print(f"{i+1}/{len(test_dataset)}: {row['agent_status']}; {len(row['retrieved_contexts'])} observed chunks")
print("Answers:", sum(r["agent_status"] == "ok" for r in results), "/", len(results))

## Evaluate with a rubric-based judge
### Exercise
Measure faithfulness to the observed contexts, relevance to the question, and helpfulness. Finish the faithfulness rubric in `evaluation_framework.py` first.

<details>
<summary>💡 NEED SOME HELP?</summary>

```python
eval_results = evaluate_rag_response(question=result["question"], response=result["agent_response"],
                                            context=context_str, judge_llm=judge_llm)
```
</details>

Scores are divided by 5 (valid range 0.2–1.0). Failed measurements stay null, and no-context faithfulness is marked not applicable. These rubric scores are not identical to the RAGAS metrics below.

In [ ]:
judge_llm = create_judge_llm()
for result in results:
    if result["agent_status"] != "ok":
        skip_scores(result, METRICS)
    else:
        context_str = "\n\n".join(result["retrieved_contexts"])
        # TODO: Evaluate the actual answer and contexts.
        eval_results = ...
        store_scores(result, eval_results)
    checkpoint(run_dir, results)
    print(result["case_id"], result["judge_status"], score_text(result["aggregate_score"]))

## Compute RAGAS metrics
RAGAS provides widely used RAG metrics. Context precision is rank-sensitive average precision: irrelevant documents after the final relevant document can leave it unchanged. Context recall measures reference claims supported by retrieval; its validity depends on the reference. Faithfulness checks response support, and answer relevancy checks alignment with the question.

### Exercise
Evaluate successful agent answers using the four metrics and the provided conservative request limits. Each metric can require several model calls; this serial pass may take tens of minutes. Missing metric values remain missing.

<details>
<summary>💡 NEED SOME HELP?</summary>

```python
ragas_results = evaluate(dataset=ragas_dataset,
            metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
            llm=ragas_judge, embeddings=embeddings,
            run_config=RunConfig(max_workers=1, timeout=600, max_retries=2))
```
</details>

In [ ]:
# Compatibility shim for the workshop's pinned RAGAS / langchain-community versions.
import types
if "langchain_community.chat_models.vertexai" not in sys.modules:
    stub = types.ModuleType("langchain_community.chat_models.vertexai")
    stub.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules["langchain_community.chat_models.vertexai"] = stub

ragas_record = {"status": "not_run", "rows": [], "attempted": len(results)}
try:
    from ragas import evaluate
    from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness, AnswerRelevancy
    from ragas.run_config import RunConfig
    from datasets import Dataset
    eligible = [r for r in results if r["agent_status"] == "ok"]
    if not eligible:
        raise ValueError("No agent answers available to evaluate")
    ragas_dataset = Dataset.from_dict({
        "question": [r["question"] for r in eligible], "answer": [r["agent_response"] for r in eligible],
        "contexts": [r["retrieved_contexts"] for r in eligible], "ground_truth": [r["ground_truth"] for r in eligible]})
    embeddings = create_embeddings()
    # Separate client: RAGAS adjusts its temperature internally.
    ragas_judge = create_judge_llm(timeout=90, max_tokens=4096)
    context_precision, context_recall = ContextPrecision(), ContextRecall()
    faithfulness, answer_relevancy = Faithfulness(), AnswerRelevancy(strictness=1)
    record_config(run_dir, ragas_embedding_model=EMBEDDING_MODEL,
                  ragas_metrics=["context_precision", "context_recall", "faithfulness", "answer_relevancy"],
                  ragas_settings={"request_timeout": 90, "metric_job_timeout": 600, "max_retries": 2,
                                  "max_workers": 1, "answer_relevancy_strictness": 1,
                                  "max_tokens": 4096, "single_generation_temperature": 1e-8})
    # TODO: Evaluate the four RAGAS metrics.
    ragas_results = ...
    frame = ragas_results.to_pandas()
    frame["case_id"] = [r["case_id"] for r in eligible]
    names = ["context_precision", "context_recall", "faithfulness", "answer_relevancy"]
    ragas_record.update(status="complete" if frame[names].notna().all().all() else "partial",
                        rows=frame.to_dict("records"), eligible=len(eligible))
    display(frame[names].agg(["mean", "count"]).T)
except Exception as exc:
    ragas_record.update(status="error", error=error_summary(exc))
    print("RAGAS did not complete:", error_summary(exc), "— check Workshop Health and saved statuses.")
checkpoint(run_dir, results, ragas=ragas_record)

## Judge calibration review
Inspect complete answers, contexts, and explanations before trusting scores. Review a varied sample (including failures, low scores, and different categories) against the same rubric, and record your own ratings independently. Three examples are a smoke check, not evidence that a judge is calibrated. A larger labeled set and agreement analysis are needed for that claim.

In [ ]:
for result in results[:3]:
    print("\nQUESTION:", result["question"], "\nSTATUS:", result["agent_status"], result["judge_status"])
    print("CONTEXTS:\n", "\n\n".join(result["retrieved_contexts"]))
    print("ANSWER:\n", result["agent_response"])
    for metric in METRICS:
        print(metric, score_text(result.get(f"{metric}_score")), result.get(f"{metric}_explanation"))
# Add your own ratings and reasons after reviewing; display is not review completion.
human_review = []
checkpoint(run_dir, results, human_review=human_review)

## Analyze measured quality and coverage
Every mean needs a denominator. A high average across a few successful cases does not hide missing answers or failed judge calls. These small instructional datasets do not establish production readiness.

In [ ]:
results_df = pd.DataFrame(results)
summary = quality_summary(results, METRICS)
print({k: v for k, v in summary.items() if k != "metrics"})
display(pd.DataFrame(summary["metrics"]).T)
print("Cases with observed KB retrieval:", sum(bool(r["retrieved_contexts"]) for r in results), "/", len(results))

## Compare categories
Category means summarize successful measurements. Check counts before comparing groups; a single example is weak evidence.

In [ ]:
display(results_df.groupby("category")[[f"{m}_score" for m in METRICS]].agg(["mean", "count"]))

## Inspect low scores and missing measurements
Use 0.6 only as a workshop triage threshold. Read the case and explanation to decide whether to improve the agent, the rubric, or the reference.

In [ ]:
for result in results:
    low = [m for m in METRICS if result.get(f"{m}_score") is not None and result[f"{m}_score"] < 0.6]
    missing = [m for m in METRICS if result.get(f"{m}_score") is None]
    if low or missing:
        print(result["question"], "\nLow:", low, "Missing:", missing)
        for metric in low + missing:
            print(" ", metric, result.get(f"{metric}_explanation"))

## Custom metric: citation validity
Check that each `[KB:source_id]` names a chunk in that answer’s actual retrieval trace. This detects invented IDs; it does not prove that the cited passage supports the adjacent claim. Inspect that relationship separately.

In [ ]:
for result in results:
    result.update(citation_check(result["agent_response"], result["retrieval_trace"]))
answered = [r for r in results if r["agent_status"] == "ok"]
print("Answers with citations that all resolve:", sum(r["all_citations_resolve"] for r in answered), "/", len(answered))
checkpoint(run_dir, results)

## Custom metric: actionability
### Exercise
Connect the custom prompt to the judge. We grade up to five successful answers to limit calls, and persist both scores and the rubric. A strict parser rejects malformed or out-of-range scores.

<details>
<summary>💡 NEED SOME HELP?</summary>

```python
custom_chain = custom_prompt | judge_llm
```
</details>

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
custom_prompt = ChatPromptTemplate.from_template("""Rate how actionable this IT answer is.
Question: {question}
Answer: {response}
1 = no useful steps; 2 = vague guidance; 3 = some usable steps; 4 = clear steps with minor gaps; 5 = complete, clear steps appropriate to the question.
Return only JSON: {{"score": <number 1-5>, "explanation": "brief reason"}}""")
# TODO: Connect the prompt and judge.
custom_chain = ...
record_config(run_dir, actionability_rubric=custom_prompt.messages[0].prompt.template)
for result in [r for r in results if r["agent_status"] == "ok"][:5]:
    try:
        judged = invoke_with_retry(lambda: custom_chain.invoke({"question": result["question"], "response": result["agent_response"]}), attempts=JUDGE_RETRY_ATTEMPTS)
        measured = parse_evaluation(judged.content, "actionability")
    except Exception as exc:
        measured = EvaluationResult(status="error", metric_name="actionability", explanation=error_summary(exc))
    result.update(actionability_score=measured.score / 5 if measured.status == "ok" else None,
                  actionability_status=measured.status, actionability_explanation=measured.explanation)
    print(result["case_id"], measured.status, score_text(result["actionability_score"]))
checkpoint(run_dir, results)

## Save this run
Each run has a unique directory, metadata manifest, full result rows, and separate RAGAS/human-review artifacts. Null scores remain distinguishable from low scores in JSON; the CSV is a convenient view.

In [ ]:
summary = quality_summary(results, METRICS)
checkpoint(run_dir, results, summary=summary)
pd.DataFrame(results).to_csv(run_dir / "results.csv", index=False)
print("Saved:", run_dir)

## Compare an improvement fairly
Change one component, restart the kernel, and rerun the same dataset. Use `compare_runs` with two saved directories; it rejects different dataset hashes, shows configuration changes, and pairs only cases successfully graded in both runs. Also compare coverage and failures. Repeat runs to assess stochastic variation.

```python
comparison = compare_runs(MODULE_DIR / "runs/rag-<before>", MODULE_DIR / "runs/rag-<after>")
print(comparison["configuration_changes"])
display(comparison["scores"])
```

In [ ]:
print("Next: inspect a failed or low-scoring case, propose one change, and rerun this saved dataset.")
print("Do not interpret an incomplete evaluation or a small score increase as deployment readiness.")

## Reflection
Which failure belongs to retrieval, generation, the reference, or the judge? Does your proposed change improve the same cases without reducing coverage? Use those observations in the continuous-improvement lesson.